### Murder in SQL City

There's been a Murder in SQL City! The SQL Murder Mystery is designed to be both a self-directed lesson to learn SQL concepts and commands and a fun game for experienced SQL users to solve an intriguing crime.

A crime has taken place and the detective needs your help. The detective gave you the crime scene report, but you somehow lost it. You vaguely remember that the crime was a ​murder​ that occurred sometime on ​Jan.15, 2018​ and that it took place in ​SQL City​. Start by retrieving the corresponding crime scene report from the police department’s database.

#### Exercise 1

Retrieve the crime scene report. Query the `crime_scene_report` table to retrieve the report. You will want to use the clues regarding the following:

- What time of crime it was
- The date it took place
- Where the crime took place


In [16]:
# Importing sqlite3 library and pprint (pretty print) function
import sqlite3
from pprint import pprint

# Creating a connectiong to the database
conn = sqlite3.connect("../data/murder_mystery.db")

# The cursor will allow us to query the database
cur = conn.cursor()
cur.execute("""
    SELECT date, city,type, description
    FROM crime_scene_report
    WHERE date = "20180115" AND type = "murder" AND city = "SQL City"
""").fetchall()

[(20180115,
  'SQL City',
  'murder',
  'Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".')]

#### Exercise 2

We will now find the witnesses described in the crime scene report. Use the information given about them to find them in the `person` table.

In [40]:

cur.execute("""
    SELECT id, name, address_street_name, address_number
    FROM person
    WHERE (name LIKE '%Annabel%' AND address_street_name = 'Franklin Ave') OR (address_street_name = 'Northwestern Dr')
    ORDER BY address_number DESC
""").fetchall()

[(14887, 'Morty Schapiro', 'Northwestern Dr', 4919),
 (17729, 'Lasonya Wildey', 'Northwestern Dr', 3824),
 (53890, 'Sophie Tiberio', 'Northwestern Dr', 3755),
 (73368, 'Torie Thalmann', 'Northwestern Dr', 3697),
 (96595, 'Coretta Cubie', 'Northwestern Dr', 3631),
 (19420, 'Cody Schiel', 'Northwestern Dr', 3524),
 (93509, 'Emmitt Aceuedo', 'Northwestern Dr', 3491),
 (87456, 'Leonora Wolfsberger', 'Northwestern Dr', 3483),
 (36378, 'Freddie Ellzey', 'Northwestern Dr', 3449),
 (53076, 'Boris Bijou', 'Northwestern Dr', 3327),
 (28360, 'Rashad Cascone', 'Northwestern Dr', 3212),
 (23044, 'Val Portlock', 'Northwestern Dr', 3143),
 (51114, 'Christena Saffell', 'Northwestern Dr', 3055),
 (68690, 'Yer Modest', 'Northwestern Dr', 3046),
 (44004, 'Alison Eska', 'Northwestern Dr', 2951),
 (98593, 'Jonah Toner', 'Northwestern Dr', 2947),
 (59307, 'Cyril Yongue', 'Northwestern Dr', 2903),
 (34155, 'Allyson Lazenson', 'Northwestern Dr', 2840),
 (90397, 'Dionna Kranwinkle', 'Northwestern Dr', 2678),
 

#### Exercise 3

We want to find the information that the witnesses provided about the crime. Use the information we have about the witnesses to find their interviews in the `interview` table.

In [49]:
cur.execute("""
    SELECT person_id, transcript
    FROM interview
    WHERE person_id IN (SELECT id 
                        FROM person
                        WHERE address_street_name = 'Northwestern Dr'
                        ORDER BY address_number DESC
                        LIMIT 1)
    OR person_id IN (SELECT id
                     FROM person
                     WHERE name ='Annabel Miller' AND address_street_name = 'Franklin Ave')
""").fetchall()

[(14887,
  'I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".'),
 (16371,
  'I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.')]

#### Exercise 4

The witnesses provided several important pieces of information about the accused:

- Part of a membership number for a gym
- Part of the license plate number for their car
- The date for another time that they visited the gym

Use the `get_fit_now_member`, `get_fit_now_check_in`, and `drivers_license` tables to get more information about the accused.

In [60]:
cur.execute("""
    SELECT get_fit_now_member.id, get_fit_now_member.name, drivers_license.plate_number, get_fit_now_check_in.check_in_date
    FROM get_fit_now_member
    JOIN get_fit_now_check_in ON get_fit_now_member.id = get_fit_now_check_in.membership_id
    JOIN person ON get_fit_now_member.person_id = person.id
    JOIN drivers_license ON person.license_id = drivers_license.id
    WHERE membership_id LIKE '48Z%' AND plate_number LIKE '%H42W%' AND check_in_date = '20180109'
    
""").fetchall()

[('48Z55', 'Jeremy Bowers', '0H42W2', 20180109)]

#### Exercise 5

You should now have a name of the individual the witnesses identified at the crime scene. You'll need to go back and look at the interview with this person found in the `interview` table.

In [62]:
cur.execute("""
    SELECT *
    FROM interview
    JOIN person ON interview.person_id = person.id
    WHERE name = 'Jeremy Bowers'
""").fetchall()

[(67318,
  'I was hired by a woman with a lot of money. I don\'t know her name but I know she\'s around 5\'5" (65") or 5\'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n',
  67318,
  'Jeremy Bowers',
  423327,
  530,
  'Washington Pl, Apt 3A',
  871539279)]

#### Exercise 6

From the interview, you've been given a description of the person that hired the individual that committed the crime. You know the following about this individual:

- Their gender
- Their height
- Their hair color
- The car they drive
- An event that they attended along with when and how often they attended the event

Using the information found in the `drivers_license` and `facebook_event_checkin` tables, pinpoint who this individual is.

In [76]:
cur.execute("""
    SELECT person.id, person.name, facebook_event_checkin.event_name, 
    facebook_event_checkin.date,
    drivers_license.hair_color,drivers_license.car_make,
    drivers_license.car_model
    FROM drivers_license
    JOIN person ON drivers_license.id = person.license_id
    JOIN facebook_event_checkin ON person.id = facebook_event_checkin.person_id
    WHERE event_name = 'SQL Symphony Concert' AND date LIKE '201712%' 
    AND hair_color = 'red' AND car_make = 'Tesla' AND car_model = 'Model S'
    GROUP BY person.id
    HAVING COUNT(*)=3
""").fetchall()

[(99716,
  'Miranda Priestly',
  'SQL Symphony Concert',
  20171206,
  'red',
  'Tesla',
  'Model S')]

#### Exercise 7

Finally, you have a name of the person who orchestrated the crime. Insert the name in the following code block where it says "mastermind_name" to see if you found the right person.

In [77]:
cur.execute("""
    INSERT INTO solution VALUES (1, "Miranda Priestly");
""")

cur.execute("""
    SELECT value FROM solution;
""").fetchall()

[('Congrats, you found the brains behind the murder! Everyone in SQL City hails you as the greatest SQL detective of all time. Time to break out the champagne!',)]